# Limpieza de Datos

## Contexto
El EDA (Notebook 01) ha revelado los siguientes problemas que este notebook 
resuelve sistemáticamente:

| Problema detectado | Solución aplicada |
|---|---|
| 42.579 juegos sin CSVs de reseñas | Merge inner JSON ↔ CSVs |
| 5.190 entradas Playtest/Demo/Beta | Filtro por sufijo en nombre |
| Juegos con < 100 reseñas | Filtro por umbral mínimo |
| 338 reseñas con texto NaN | Eliminación directa |
| Reseñas con < 10 caracteres (spam) | Filtro por longitud |
| `early_access_review` con 97.87% nulos | Eliminación de columna |
| `recommend` como texto | Conversión a binario (1/0) |
| `post_date` como string | Parseo a datetime |
| Outliers de precio (software profesional ≥100€) | Flag `is_software_tool` |
| Columnas multicolineales (corr > 0.85) | Eliminación selectiva |
| 31M de reseñas sin granularidad manejable | Guardado en Parquet por chunks |

## Estructura del notebook

### Bloque A — Limpieza del JSON (`df_games`)
- A1. Eliminar Playtests, Demos y Betas
- A2. Merge inner con CSVs disponibles
- A3. Filtrar juegos con ≥ 100 reseñas
- A4. Limpiar tipos de datos, outliers y columnas redundantes
- A5. Encoding de géneros, categorías y tags
- A6. Guardar `games_clean.parquet`

### Bloque B — Limpieza de los CSVs de reseñas
- B1. Procesado por chunks sobre los ~11.500 CSVs elegibles
- B2. Filtros de calidad del texto (NaN, < 10 chars)
- B3. Eliminar columna `early_access_review`
- B4. Convertir `recommend` → binario
- B5. Parsear `post_date` → datetime
- B6. Guardar `reviews_clean.parquet`

### Bloque C — Construcción de la variable objetivo
- C1. Calcular `total_reviews` y `positive_ratio` por juego
- C2. Aplicar umbrales: ≥ 50 reseñas AND ≥ 80% positivos y separación temporal
- C3. Guardar `target.parquet`

### Bloque D — Integración de Twitch
- D1. Filtrar Twitch desde 2020
- D2. Normalizar nombres y cruzar con Steam
- D3. Crear feature binaria `appeared_on_twitch_top200`
- D4. Guardar `twitch_features.parquet`

## Output esperado
Al finalizar este notebook tendremos 4 archivos Parquet en `data/processed/`:
- `games_clean.parquet` → metadatos limpios de ~11.500 juegos
- `reviews_clean.parquet` → reseñas limpias con fecha y etiqueta binaria
- `target.parquet` → variable objetivo `is_successful` por juego
- `twitch_features.parquet` → feature de presencia en Twitch por juego



# Bloque A: Limpieza del JSON (df_games)

In [1]:
import pandas as pd
import numpy as np
import json
import os
import re
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter

# ── Rutas ──────────────────────────────────────────────
JSON_PATH   = Path('../data/raw/games.json')
REVIEWS_DIR = Path('../data/raw/reviews/')
PROCESSED   = Path('../data/processed/')
PROCESSED.mkdir(exist_ok=True)

# ── Carga del JSON ─────────────────────────────────────
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    games_raw = json.load(f)

df_games = pd.DataFrame.from_dict(games_raw, orient='index').reset_index()
df_games = df_games.rename(columns={'index': 'appid'})

print(f" JSON cargado: {len(df_games):,} juegos")


 JSON cargado: 65,686 juegos


In [2]:
# Eliminar juegos demo, beta, etc.
mask_aux = df_games['name'].str.contains(
    r'\bPlaytest\b|\bDemo\b|\bBeta\b|\bTest Server\b|\bTrials?\b',
    case=False, regex=True, na=False
)

print(f"Entradas Playtest/Demo/Beta eliminadas: {mask_aux.sum():,}")
df_games = df_games[~mask_aux].copy()
print(f"Juegos restantes: {len(df_games):,}")


Entradas Playtest/Demo/Beta eliminadas: 5,130
Juegos restantes: 60,556


In [3]:
# Extraer appids y recuento de reseñas desde nombres de CSV
csv_files = list(REVIEWS_DIR.glob('*.csv'))

reviews_count = {}
for f in csv_files:
    parts = f.stem.split('_')
    if len(parts) == 2:
        try:
            reviews_count[parts[0]] = int(parts[1])
        except ValueError:
            continue

df_csv_counts = pd.DataFrame(
    reviews_count.items(), 
    columns=['appid', 'csv_review_count']
)

print(f"CSVs disponibles: {len(df_csv_counts):,}")

# Merge inner: solo juegos con JSON + CSV
df_games = df_games.merge(df_csv_counts, on='appid', how='inner')
print(f"Juegos con JSON + CSV: {len(df_games):,}")

# Filtro ≥ 100 reseñas
df_games = df_games[df_games['csv_review_count'] >= 100].copy()
print(f"Juegos con ≥ 100 reseñas (dataset final): {len(df_games):,}")


CSVs disponibles: 23,107
Juegos con JSON + CSV: 22,951
Juegos con ≥ 100 reseñas (dataset final): 11,507


In [4]:
# 1. release_date → datetime
df_games['release_date'] = pd.to_datetime(
    df_games['release_date'], format='mixed', errors='coerce'
)
print(f"Fechas no parseables: {df_games['release_date'].isna().sum()}")

# 2. Columnas a eliminar (redundantes o sin valor predictivo)
cols_to_drop = [
    'detailed_description',
    'about_the_game', 
    'average_playtime_2weeks',
    'median_playtime_2weeks',
    'average_playtime_forever',  # Alta correlación con median (0.88)
    'discount',
    'positive',   # Se recalculará desde CSVs
    'negative',   # Se recalculará desde CSVs
]
cols_to_drop = [c for c in cols_to_drop if c in df_games.columns]
df_games = df_games.drop(columns=cols_to_drop)
print(f"Columnas eliminadas: {cols_to_drop}")

# 3. estimated_owners → punto medio numérico
def parsear_owners(s):
    if not isinstance(s, str):
        return np.nan
    partes = s.replace(',', '').split(' - ')
    try:
        return (int(partes[0]) + int(partes[1])) / 2
    except:
        return np.nan

df_games['owners_midpoint'] = df_games['estimated_owners'].apply(parsear_owners)

# 4. Outliers de precio: software profesional
df_games['is_software_tool'] = (df_games['price'] >= 100).astype(int)
df_games['price_clean'] = df_games['price'].clip(upper=99.99)
print(f"Juegos marcados como software tool (precio ≥ 100€): {df_games['is_software_tool'].sum()}")

# 5. Variables binarias útiles
df_games['is_free'] = (df_games['price'] == 0).astype(int)
df_games['has_full_audio'] = df_games['full_audio_languages'].apply(
    lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0
)

# 6. Número de idiomas y categorías
df_games['num_languages'] = df_games['supported_languages'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)
df_games['num_categories'] = df_games['categories'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print(f"\nShape tras limpieza de columnas: {df_games.shape}")


Fechas no parseables: 0
Columnas eliminadas: ['detailed_description', 'about_the_game', 'average_playtime_2weeks', 'median_playtime_2weeks', 'average_playtime_forever', 'discount', 'positive', 'negative']
Juegos marcados como software tool (precio ≥ 100€): 3

Shape tras limpieza de columnas: (11507, 22)


In [5]:
# ── Géneros (MultiLabelBinarizer) ─────────────────────
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(
    df_games['genres'].apply(lambda x: x if isinstance(x, list) else [])
)
df_genres = pd.DataFrame(
    genres_encoded,
    columns=[f'genre_{g.lower().replace(" ", "_").replace("&", "and")}' 
             for g in mlb.classes_],
    index=df_games.index
)
print(f"Columnas de géneros generadas: {len(df_genres.columns)}")
print(df_genres.columns.tolist())

# ── Top 20 Tags más frecuentes (del EDA) ──────────────
all_tags = []
for tags_dict in df_games['tags'].dropna():
    if isinstance(tags_dict, dict):
        all_tags.extend(tags_dict.keys())

top_tags = [tag for tag, _ in Counter(all_tags).most_common(20)]

for tag in top_tags:
    col = f'tag_{tag.lower().replace(" ", "_").replace("-", "_")}'
    df_games[col] = df_games['tags'].apply(
        lambda x: 1 if isinstance(x, dict) and tag in x else 0
    )
print(f"Columnas de tags generadas: 20")

# ── Concatenar géneros ─────────────────────────────────
df_games = pd.concat(
    [df_games.reset_index(drop=True), df_genres.reset_index(drop=True)], 
    axis=1
)
print(f"\nShape final con géneros y tags: {df_games.shape}")


Columnas de géneros generadas: 23
['genre_accounting', 'genre_action', 'genre_adventure', 'genre_animation_and_modeling', 'genre_audio_production', 'genre_casual', 'genre_design_and_illustration', 'genre_early_access', 'genre_education', 'genre_free_to_play', 'genre_game_development', 'genre_indie', 'genre_massively_multiplayer', 'genre_photo_editing', 'genre_rpg', 'genre_racing', 'genre_simulation', 'genre_software_training', 'genre_sports', 'genre_strategy', 'genre_utilities', 'genre_video_production', 'genre_web_publishing']
Columnas de tags generadas: 20

Shape final con géneros y tags: (11507, 65)


In [6]:
# Columnas a eliminar antes de guardar (listas originales ya procesadas)
cols_raw = [
    'genres', 'categories', 'tags', 
    'supported_languages', 'full_audio_languages',
    'estimated_owners', 'price', 'short_description'
]
cols_raw = [c for c in cols_raw if c in df_games.columns]
df_games_clean = df_games.drop(columns=cols_raw).copy()

# Guardar
df_games_clean.to_parquet(PROCESSED / 'games_clean.parquet', index=False)

size_mb = (PROCESSED / 'games_clean.parquet').stat().st_size / 1024 / 1024
print(f"   Juegos: {len(df_games_clean):,}")
print(f"   Columnas: {len(df_games_clean.columns)}")
print(f"   Tamaño: {size_mb:.2f} MB")
print(f"\nColumnas finales:")
print(df_games_clean.columns.tolist())


   Juegos: 11,507
   Columnas: 57
   Tamaño: 0.50 MB

Columnas finales:
['appid', 'name', 'release_date', 'required_age', 'median_playtime_forever', 'peak_ccu', 'csv_review_count', 'owners_midpoint', 'is_software_tool', 'price_clean', 'is_free', 'has_full_audio', 'num_languages', 'num_categories', 'tag_singleplayer', 'tag_indie', 'tag_adventure', 'tag_action', 'tag_casual', 'tag_2d', 'tag_simulation', 'tag_story_rich', 'tag_rpg', 'tag_atmospheric', 'tag_3d', 'tag_exploration', 'tag_strategy', 'tag_cute', 'tag_multiplayer', 'tag_first_person', 'tag_colorful', 'tag_anime', 'tag_fantasy', 'tag_pixel_graphics', 'genre_accounting', 'genre_action', 'genre_adventure', 'genre_animation_and_modeling', 'genre_audio_production', 'genre_casual', 'genre_design_and_illustration', 'genre_early_access', 'genre_education', 'genre_free_to_play', 'genre_game_development', 'genre_indie', 'genre_massively_multiplayer', 'genre_photo_editing', 'genre_rpg', 'genre_racing', 'genre_simulation', 'genre_software_

# Bloque B: Limpieza de datos CSVs

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

REVIEWS_DIR = Path('../data/raw/reviews/')
PROCESSED   = Path('../data/processed/')

# Cargar los appids elegibles del Bloque A
df_games_clean = pd.read_parquet(PROCESSED / 'games_clean.parquet')
eligible_appids = set(df_games_clean['appid'].astype(str).tolist())

# Filtrar solo los CSVs de juegos elegibles
all_csvs = list(REVIEWS_DIR.glob('*.csv'))
eligible_csvs = [
    f for f in all_csvs 
    if f.stem.split('_')[0] in eligible_appids
]

print(f"Total CSVs disponibles: {len(all_csvs):,}")
print(f"CSVs de juegos elegibles: {len(eligible_csvs):,}")
print(f"CSVs descartados (juegos no elegibles): {len(all_csvs) - len(eligible_csvs):,}")


Total CSVs disponibles: 23,107
CSVs de juegos elegibles: 11,507
CSVs descartados (juegos no elegibles): 11,600


Como hay aproximadamente 31M de reviews, no es factible cargarlos todos en memoria. Por lo tanto, se va a cargar un subconjunto de los reviews para realizar la limpieza. Para ello, se va a seleccionar un subconjunto de las 500 reviews con mejor helpfulness. En el caso de que tenga menos de 500 reviews, se seleccionarán todas.
Asi mismo, en vez de tratar los CSV como uno solo, se van a iterar sobre cada uno de ellos e ir aplicando la limpieza a cada uno de los archivos.

In [8]:
def limpiar_csv(filepath, max_reviews=500):
    """
    Carga, limpia y submuestrea un CSV de reseñas.
    Devuelve un DataFrame limpio o None si hay error.
    """
    try:
        df = pd.read_csv(
            filepath, 
            encoding='utf-8', 
            encoding_errors='replace',
            dtype={'helpfulness': 'float32'}
        )
        
        appid = filepath.stem.split('_')[0]
        df['appid'] = appid
        
        # 1. Eliminar columna early_access_review
        if 'early_access_review' in df.columns:
            df = df.drop(columns=['early_access_review'])
        
        # 2. Eliminar reseñas con texto NaN
        df = df.dropna(subset=['review'])
        
        # 3. Eliminar reseñas con < 10 caracteres (spam/vacías)
        df = df[df['review'].astype(str).str.len() >= 10]
        
        # 4. Convertir recommend a binario
        df['recommend'] = df['recommend'].map({
            'Recommended': 1, 
            'Not Recommended': 0
        })
        df = df.dropna(subset=['recommend'])
        df['recommend'] = df['recommend'].astype(int)
        
        # 5. Parsear post_date a datetime
        df['post_date'] = pd.to_datetime(
            df['post_date'], format='mixed', errors='coerce'
        )
        
        # 6. Limpiar helpfulness
        df['helpfulness'] = pd.to_numeric(df['helpfulness'], errors='coerce').fillna(0)
        
        # 7. Submuestreo: top N por helpfulness (o todas si hay menos de N)
        if len(df) > max_reviews:
            df = df.nlargest(max_reviews, 'helpfulness')
        
        # 8. Añadir longitud del texto como feature temprana
        df['review_len_chars'] = df['review'].astype(str).str.len()
        
        return df
    
    except Exception as e:
        print(f"Error en {filepath.name}: {e}")
        return None


In [9]:
# Probar con 5 CSVs para verificar que la función funciona correctamente
test_csvs = eligible_csvs[:5]
test_results = []

for csv in test_csvs:
    result = limpiar_csv(csv)
    if result is not None:
        test_results.append(result)
        print(f"{csv.name}: {len(result)} reseñas --> OK")

df_test = pd.concat(test_results, ignore_index=True)
print(f"\n--- Resultado del test ---")
print(f"Shape: {df_test.shape}")
print(f"Columnas: {df_test.columns.tolist()}")
print(f"Tipos:\n{df_test.dtypes}")
print(f"\nNulos:\n{df_test.isnull().sum()}")
print(f"\nEjemplo de filas:")
print(df_test.head(3))


1000030_2098.csv: 500 reseñas --> OK
1000360_6765.csv: 500 reseñas --> OK
1000410_2404.csv: 500 reseñas --> OK
1000440_407.csv: 336 reseñas --> OK
1000700_250.csv: 240 reseñas --> OK

--- Resultado del test ---
Shape: (2076, 8)
Columnas: ['user', 'playtime', 'post_date', 'helpfulness', 'review', 'recommend', 'appid', 'review_len_chars']
Tipos:
user                           str
playtime                   float64
post_date           datetime64[us]
helpfulness                float32
review                         str
recommend                    int64
appid                          str
review_len_chars             int64
dtype: object

Nulos:
user                0
playtime            0
post_date           0
helpfulness         0
review              0
recommend           0
appid               0
review_len_chars    0
dtype: int64

Ejemplo de filas:
         user  playtime  post_date  helpfulness  \
0      lazEar       8.1 2020-03-06        328.0   
1      Kirito       5.1 2021-07-10        

In [10]:
BATCH_SIZE = 500  # Guardar en Parquet cada N CSVs procesados
output_path = PROCESSED / 'reviews_clean.parquet'

writer = None
schema = None

errores = []
total_reviews_escritas = 0
total_csvs_procesados = 0

with tqdm(total=len(eligible_csvs), desc="Procesando CSVs") as pbar:
    batch = []
    
    for i, csv_path in enumerate(eligible_csvs):
        df_clean = limpiar_csv(csv_path)
        
        if df_clean is not None and len(df_clean) > 0:
            batch.append(df_clean)
            total_reviews_escritas += len(df_clean)
            total_csvs_procesados += 1
        else:
            errores.append(csv_path.name)
        
        # Cada BATCH_SIZE CSVs, volcar a Parquet y liberar memoria
        if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(eligible_csvs):
            if batch:
                df_batch = pd.concat(batch, ignore_index=True)
                table = pa.Table.from_pandas(df_batch, preserve_index=False)
                
                if writer is None:
                    schema = table.schema
                    writer = pq.ParquetWriter(output_path, schema)
                
                writer.write_table(table)
                batch = []
                del df_batch
        
        pbar.update(1)
        pbar.set_postfix({
            'reseñas': f'{total_reviews_escritas:,}',
            'errores': len(errores)
        })

if writer:
    writer.close()

print(f"\n Proceso completado")
print(f"   CSVs procesados: {total_csvs_procesados:,}")
print(f"   Reseñas escritas: {total_reviews_escritas:,}")
print(f"   Errores: {len(errores)}")
if errores:
    print(f"   CSVs con error: {errores[:10]}")

size_mb = output_path.stat().st_size / 1024 / 1024
print(f"   Tamaño reviews_clean.parquet: {size_mb:.1f} MB")


Procesando CSVs: 100%|██████████| 11507/11507 [08:23<00:00, 22.87it/s, reseñas=3,745,380, errores=0]


 Proceso completado
   CSVs procesados: 11,507
   Reseñas escritas: 3,745,380
   Errores: 0
   Tamaño reviews_clean.parquet: 1415.0 MB


In [11]:
# Cargar una muestra para verificar que el Parquet está bien
df_verify = pd.read_parquet(PROCESSED / 'reviews_clean.parquet')

print(f" Verificación de reviews_clean.parquet")
print(f"   Shape: {df_verify.shape}")
print(f"   Juegos únicos: {df_verify['appid'].nunique():,}")
print(f"   Rango de fechas: {df_verify['post_date'].min()} → {df_verify['post_date'].max()}")
print(f"\nDistribución de recommend:")
print(df_verify['recommend'].value_counts())
print(f"\nNulos:")
print(df_verify.isnull().sum())
print(f"\nEjemplo:")
print(df_verify.head(3))


 Verificación de reviews_clean.parquet
   Shape: (3745380, 8)
   Juegos únicos: 11,507
   Rango de fechas: 2011-04-17 00:00:00 → 2024-12-06 00:00:00

Distribución de recommend:
recommend
1    2886620
0     858760
Name: count, dtype: int64

Nulos:
user                734
playtime            224
post_date             0
helpfulness           0
review                0
recommend             0
appid                 0
review_len_chars      0
dtype: int64

Ejemplo:
         user  playtime  post_date  helpfulness  \
0      lazEar       8.1 2020-03-06        328.0   
1      Kirito       5.1 2021-07-10        193.0   
2  Mr_LeonLau       2.0 2023-05-24        179.0   

                                              review  recommend    appid  \
0                                        chinese,pls          1  1000030   
1                                    We need Chinese          1  1000030   
2  游戏本身是好游戏，我就想问问Vertigo，你把台湾单独列出来是Taiwan我忍了，蒸馍馍是...          0  1000030   

   review_len_chars  
0     

Nulos en user (734) y playtime (224): son irrelevantes. user no será feature del modelo y playtime tiene tan pocos nulos que se imputarán con 0 en el preprocesamiento. 

Fechas desde 2011: algunos juegos tienen reseñas muy antiguas. Esto no es un problema porque el filtro temporal para las features (+30 días desde el lanzamiento) lo resolverá automáticamente en el Feature Engineering.


# Bloque C: Construccion de variable Objetivo

Separamos la variable target desde los 30 días hasta los 6 meses, para evitar data leakage

In [12]:
df_reviews = pd.read_parquet(PROCESSED / 'reviews_clean.parquet')
df_games   = pd.read_parquet(PROCESSED / 'games_clean.parquet')

# Cruzar reseñas con release_date del juego
df_reviews = df_reviews.merge(
    df_games[['appid', 'release_date']], 
    on='appid', 
    how='left'
)
# Calcular días desde el lanzamiento
df_reviews['days_since_release'] = (
    df_reviews['post_date'] - df_reviews['release_date']
).dt.days


df_future = df_reviews[
    (df_reviews['days_since_release'] > 30) & 
    (df_reviews['days_since_release'] <= 180) # Éxito a 6 meses
]

# Agregar por juego: total de reseñas y ratio de positivos
df_target = df_future.groupby('appid').agg(
    total_reviews  = ('recommend', 'count'),
    total_positive = ('recommend', 'sum'),
).reset_index()

df_target['positive_ratio'] = df_target['total_positive'] / df_target['total_reviews']

print(f"Juegos en el target: {len(df_target):,}")
print(f"\nEstadísticas:")
print(df_target[['total_reviews', 'positive_ratio']].describe())


Juegos en el target: 11,371

Estadísticas:
       total_reviews  positive_ratio
count    11371.00000    11371.000000
mean        66.50277        0.767384
std         51.81031        0.188542
min          1.00000        0.000000
25%         29.00000        0.666667
50%         54.00000        0.812500
75%         89.00000        0.911765
max        491.00000        1.000000


In [13]:
# Umbrales definidos y justificados en el EDA:
# ≥ 100 reseñas (tras submuestreo) AND ≥ 90% positivos
UMBRAL_REVIEWS  = 50
UMBRAL_POSITIVO = 0.80

df_target['is_successful'] = (
    (df_target['total_reviews'] >= UMBRAL_REVIEWS) &
    (df_target['positive_ratio'] >= UMBRAL_POSITIVO)
).astype(int)

conteo = df_target['is_successful'].value_counts()
print(f"=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ===")
print(f"No exitosos (0): {conteo[0]:,} ({conteo[0]/len(df_target)*100:.1f}%)")
print(f"Exitosos    (1): {conteo[1]:,} ({conteo[1]/len(df_target)*100:.1f}%)")
print(f"Ratio desbalance: {conteo[0]/conteo[1]:.1f}:1")


=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ===
No exitosos (0): 8,072 (71.0%)
Exitosos    (1): 3,299 (29.0%)
Ratio desbalance: 2.4:1


In [14]:
df_target.to_parquet(PROCESSED / 'target.parquet', index=False)

size_mb = (PROCESSED / 'target.parquet').stat().st_size / 1024 / 1024
print(f"   Juegos: {len(df_target):,}")
print(f"   Tamaño: {size_mb:.2f} MB")
print(f"\nMuestra:")
print(df_target.head(10))


   Juegos: 11,371
   Tamaño: 0.14 MB

Muestra:
     appid  total_reviews  total_positive  positive_ratio  is_successful
0  1000030             51              43        0.843137              1
1  1000360             48              44        0.916667              0
2  1000410             49              12        0.244898              0
3  1000440             40              37        0.925000              0
4  1000700             46              44        0.956522              0
5  1000760             66              45        0.681818              0
6  1000770             14               5        0.357143              0
7  1001270            261             202        0.773946              0
8  1001800             77              60        0.779221              0
9  1001860             15              14        0.933333              0


# Bloque D: integración de Twitch

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path('../data/processed/')

# Cargar datos
df_twitch = pd.read_csv('../data/raw/Twitch_game_data.csv', encoding='latin1')
df_games_clean = pd.read_parquet(PROCESSED / 'games_clean.parquet')

# Filtrar solo desde 2020 (periodo relevante para nuestros juegos)
df_twitch = df_twitch[df_twitch['Year'] >= 2020].copy()

print(f"Registros Twitch desde 2020: {len(df_twitch):,}")
print(f"Juegos únicos en Twitch (2020+): {df_twitch['Game'].nunique():,}")


Registros Twitch desde 2020: 11,400
Juegos únicos en Twitch (2020+): 1,341


In [16]:
def normalizar_nombre(nombre):
    if not isinstance(nombre, str):
        return ''
    nombre = nombre.lower().strip()
    nombre = re.sub(r'[^\w\s]', '', nombre)  # Eliminar puntuación
    nombre = re.sub(r'\s+', ' ', nombre)      # Espacios múltiples
    return nombre.strip()

# Normalizar nombres en ambos datasets
df_twitch['game_normalized'] = df_twitch['Game'].apply(normalizar_nombre)
df_games_clean['name_normalized'] = df_games_clean['name'].apply(normalizar_nombre)

# Ver ejemplos de normalización
print("Ejemplos de normalización:")
for orig, norm in zip(df_twitch['Game'].head(5), df_twitch['game_normalized'].head(5)):
    print(f"  '{orig}' --> '{norm}'")


Ejemplos de normalización:
  'League of Legends' --> 'league of legends'
  'Just Chatting' --> 'just chatting'
  'Escape From Tarkov' --> 'escape from tarkov'
  'Fortnite' --> 'fortnite'
  'Dota 2' --> 'dota 2'


In [17]:
# Convertir Year y Month a fecha
df_twitch['twitch_date'] = pd.to_datetime(
    df_twitch[['Year','Month']].assign(Day=1)
)

# Cruzar con release_date de cada juego
df_merged = df_twitch.merge(
    df_games[['appid','name','release_date']], 
    left_on='Game', right_on='name', 
    how='inner'
)

# Calcular mes de lanzamiento del juego
df_merged['release_month'] = df_merged['release_date'].dt.to_period('M')
df_merged['twitch_month']  = df_merged['twitch_date'].dt.to_period('M')

# Filtrar SOLO el mes de lanzamiento 
df_launch = df_merged[
    df_merged['twitch_month'] == df_merged['release_month']
]



In [18]:
# Agregar métricas de Twitch por juego (máximo histórico desde 2020)
df_twitch_agg = df_launch.groupby('game_normalized').agg(
    twitch_peak_viewers     = ('Peak_viewers', 'max'),
    twitch_hours_watched    = ('Hours_watched', 'sum'),
    twitch_avg_viewers      = ('Avg_viewers', 'max'),
    twitch_months_in_top200 = ('Game', 'count')
).reset_index()

# Cruce con Steam por nombre normalizado
df_twitch_merged = df_games_clean[['appid', 'name_normalized']].merge(
    df_twitch_agg,
    left_on='name_normalized',
    right_on='game_normalized',
    how='left'
)

# Feature binaria: ¿apareció en el Top 200 de Twitch?
df_twitch_merged['appeared_on_twitch_top200'] = (
    df_twitch_merged['twitch_peak_viewers'].notna()
).astype(int)

# Para juegos sin match, rellenar métricas con 0
cols_twitch = [
    'twitch_peak_viewers', 'twitch_hours_watched', 
    'twitch_avg_viewers', 'twitch_months_in_top200'
]
df_twitch_merged[cols_twitch] = df_twitch_merged[cols_twitch].fillna(0)

# Verificar resultado
matches = df_twitch_merged['appeared_on_twitch_top200'].sum()
print(f"Juegos con match en Twitch: {matches:,} ({matches/len(df_twitch_merged)*100:.1f}%)")
print(f"Juegos sin match (Twitch = 0): {len(df_twitch_merged) - matches:,}")
print(f"\nEstadísticas de métricas Twitch (juegos con match):")
print(df_twitch_merged[df_twitch_merged['appeared_on_twitch_top200']==1][cols_twitch].describe())


Juegos con match en Twitch: 356 (3.1%)
Juegos sin match (Twitch = 0): 11,151

Estadísticas de métricas Twitch (juegos con match):
       twitch_peak_viewers  twitch_hours_watched  twitch_avg_viewers  \
count         3.560000e+02          3.560000e+02          356.000000   
mean          9.167043e+04          4.568646e+06         6315.087079   
std           1.333460e+05          1.134893e+07        16273.135569   
min           4.682000e+03          3.782560e+05          544.000000   
25%           3.379700e+04          9.254378e+05         1251.500000   
50%           5.228950e+04          1.519559e+06         2066.500000   
75%           9.274175e+04          3.889296e+06         5255.500000   
max           1.273854e+06          1.459829e+08       217560.000000   

       twitch_months_in_top200  
count               356.000000  
mean                  1.011236  
std                   0.105551  
min                   1.000000  
25%                   1.000000  
50%                   1

In [19]:
df_twitch_features = df_twitch_merged[
    ['appid', 'appeared_on_twitch_top200'] + cols_twitch
].copy()

df_twitch_features.to_parquet(PROCESSED / 'twitch_features.parquet', index=False)

size_mb = (PROCESSED / 'twitch_features.parquet').stat().st_size / 1024 / 1024
print(f" twitch_features.parquet guardado")
print(f"   Juegos: {len(df_twitch_features):,}")
print(f"   Tamaño: {size_mb:.2f} MB")
print(f"\nMuestra:")
print(df_twitch_features[df_twitch_features['appeared_on_twitch_top200']==1].head(5))


 twitch_features.parquet guardado
   Juegos: 11,507
   Tamaño: 0.10 MB

Muestra:
       appid  appeared_on_twitch_top200  twitch_peak_viewers  \
2     999220                          1             244510.0   
57   1681430                          1              29966.0   
64   1496790                          1              80670.0   
92   1519350                          1              32817.0   
150  2448970                          1              88713.0   

     twitch_hours_watched  twitch_avg_viewers  twitch_months_in_top200  
2               3963640.0              5334.0                      1.0  
57              1298060.0              1805.0                      1.0  
64              2745528.0              3695.0                      1.0  
92              3050125.0              4105.0                      1.0  
150             5230702.0              7039.0                      1.0  


##  Resumen final

### Archivos generados en `data/processed/`
| Archivo | Registros | Columnas | Tamaño |
|---|---|---|---|
| `games_clean.parquet` | 11.507 juegos | 57 | ~0.5 MB |
| `reviews_clean.parquet` | 3.745.380 reseñas | 8 | ~600 MB |
| `target.parquet` | 11.507 juegos | 5 | ~0.2 MB |
| `twitch_features.parquet` | 11.507 juegos | 6 | ~0.1 MB |

### Próximo paso: Notebook 03 — Feature Engineering
Con los datos limpios y estructurados, el siguiente notebook construirá:
- **Features NLP** desde el texto de reseñas (sentimiento, longitud, bugs)
  aplicando el filtro temporal T+30 días para evitar data leakage
- **Variable objetivo** ya disponible en `target.parquet`
- **Dataset maestro** uniendo las 4 fuentes en un único DataFrame listo 
  para el modelado
